# Using the Python API

In this interactive Jupyter notebook, you will learn how to initialize a simple GLORYxR predictor from an existing set of metabolism reactions and a FAME3R-based scoring model. In addition, we illustrate the advantages of the newly introduced "strict SOM annotation" mode and show how you can customize GLORYxR to your own needs.

In [ ]:
from IPython.display import display

In [ ]:
from rdkit.Chem.rdmolfiles import MolFromSmiles

from gloryxr import GLORYxR, Reactor
from gloryxr.models.fame3r import SingleFAME3RModelProvider

## Initializing the predictor

GLORYxR predictors are made up of a {class}`~gloryxr.reactions.Reactor` reaction engine which applies reaction rules and a {class}`~gloryxr.models.ModelProvider` which then scores the resulting reactions.
As such, we initialize the {class}`~gloryxr.GLORYxR` predictor by providing both.

In [ ]:
gloryxr = GLORYxR(
    models=SingleFAME3RModelProvider.load("example_model.joblib"),
    reactor=Reactor.load_builtin(phase="1+2", strict_soms=False),
)

```{warning}
Please note that the model used in this example is NOT trained on the full GLORYxR dataset!
Instead it is a rudimentary model that is used only in this tutorial in order to illustrate how to use the metabolite prediction API.
```

````{tip}
If you want to work with a model that is maximally compatible with the original GLORYx, download the required model dumps from [our Zenodo archive](https://example.com) and initialize your predictor like so:

```python
gloryxr = GLORYxR(
    models=MultiFAME3RModelProvider.load("PATH_TO_MODELS"),
    reactor=Reactor.load_builtin(phase="1+2", strict_soms=False),
)
```
````

## Making predictions

We can now use the {class}`~gloryxr.GLORYxR` instance to predict metabolic reactions for a molecule.

For example, here we predict possible reactions involving the drug Paracetamol.

In [ ]:
paracetamol = MolFromSmiles("CC(=O)Nc1ccc(O)cc1")
paracetamol

In [ ]:
predicted_reactions = gloryxr.predict_one(paracetamol)

## Inspecting predictions

We can now inspect one of the predicted reactions. Note that the ordering of reactions in the returned output is arbitrary.
As such, the metabolism reaction at index 0 is not the most likely, but instead just one of the returned reactions.

When displaying the reaction, we can see that some atoms are highlighted. Highlighting indicates that the atom was passed to the underlying model for scoring. The default models return the highest score among the considered atoms as the score of the reaction.

In [ ]:
predicted_reactions[0]

It is also possible to retrieve the score from the predicted reaction.

In [ ]:
predicted_reactions[0].GetDoubleProp("Score")

If we are interested in viewing the most likely metabolic reaction this is easily possible.

In [ ]:
predicted_reactions_sorted = sorted(
    predicted_reactions, key=lambda rxn: rxn.GetDoubleProp("Score"), reverse=True
)

With this setup, the most relevant metabolism pathways (O-sulfation, O-glucuronidation and oxidation to NAPQI) are all captured in the top 5 predicted reactions. However, because of the promiscuous atom matching of the applied rules, we can also see that an additional aromatic hydroxylation reaction has been scored highly, which is unexpected.

In [ ]:
display(*predicted_reactions_sorted[:5])

## Strict SOM annotation

To solve the issue of reactions receiving inflated scores because of promiscuous atom matching, the "strict SOM annotation" mode is available. With this mode enabled, the {class}`~gloryxr.reactions.Reactor` uses various heuristics to restrict the SOM to the most relevant atom or atoms.

The strict SOM mode can be enabled by passing `strict_soms=True` to any of the {class}`~gloryxr.reactions.Reactor` initialization functions.

In [ ]:
gloryxr_strict = GLORYxR(
    models=SingleFAME3RModelProvider.load("example_model.joblib"),
    reactor=Reactor.load_builtin(phase="1+2", strict_soms=True),
)

With this setup, the most relevant metabolism pathways (O-sulfation, O-glucuronidation and oxidation to NAPQI) are again all captured in the top 5 predicted reactions. However, the aromatic hydroxylation reaction, which scored highly in the non-strict SOM mode, has disappeared.

In [ ]:
display(
    *sorted(
        gloryxr_strict.predict_one(paracetamol),
        key=lambda rxn: rxn.GetDoubleProp("Score"),
        reverse=True,
    )[:5]
)